# Libraries

In [2]:
import pandas as pd

# Data

## Air Pollution Data

In [3]:
def read_files(file_paths: list, drop_columns: list):
    dfs = [
        pd.read_csv(file).assign(year=int(file[-8:-4])).drop(columns=drop_columns)
        for file in file_paths
    ]

    df = pd.concat(dfs, ignore_index=True)

    return df

In [ ]:
filenames_pollutants = ["data/ugz_ogd_air_d1_2023.csv", "data/ugz_ogd_air_d1_2024.csv"]

#df_pollutants = read_files(file_paths=filenames_pollutants, drop_columns=["Status", "Intervall"])

# Creating pandas dataframe instead of using the function read_files
df_pollutants = pd.concat(
    [pd.read_csv(f).drop(columns=["Status", "Intervall"]) for f in filenames_pollutants],
    ignore_index=True
)

df_pollutants

,Datum,Standort,Parameter,Einheit,Wert
0,2023-01-01T00:00+0100,Zch_Stampfenbachstrasse,SO2,µg/m3,1.51
1,2023-01-01T00:00+0100,Zch_Stampfenbachstrasse,CO,mg/m3,0.35
2,2023-01-01T00:00+0100,Zch_Stampfenbachstrasse,NOx,ppb,27.13
3,2023-01-01T00:00+0100,Zch_Stampfenbachstrasse,NO,µg/m3,16.57
4,2023-01-01T00:00+0100,Zch_Stampfenbachstrasse,NO2,µg/m3,26.47
...,...,...,...,...,...
25581,2024-12-31T00:00+0100,Zch_Heubeeribüel,O3,µg/m3,13.42
25582,2024-12-31T00:00+0100,Zch_Heubeeribüel,PM10,µg/m3,13.75
25583,2024-12-31T00:00+0100,Zch_Heubeeribüel,PM2.5,µg/m3,13.36
25584,2024-12-31T00:00+0100,Zch_Heubeeribüel,O3_max_h1,µg/m3,27.65


In [35]:

# Display df_pollutants info and missing values
print("df_pollutants shape:", df_pollutants.shape)
print("\nMissing values per column:")
print(df_pollutants.isnull().sum())
print("\nData types:")
print(df_pollutants.dtypes)


df_pollutants shape: (25586, 5)

Missing values per column:
Datum          0
Standort       0
Parameter      0
Einheit        0
Wert         858
dtype: int64

Data types:
Datum         object
Standort      object
Parameter     object
Einheit       object
Wert         float64
dtype: object


In [22]:
param_units_pollutants = (
    df_pollutants[["Parameter", "Einheit"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

param_units_pollutants

,Parameter,Einheit
0,SO2,µg/m3
1,CO,mg/m3
2,NOx,ppb
3,NO,µg/m3
4,NO2,µg/m3
5,O3,µg/m3
6,PM10,µg/m3
7,PM2.5,µg/m3
8,O3_max_h1,µg/m3
9,O3_nb_h1>120,1


In [63]:
# Pivot with split date and time columns

# 1) Parse 'Datum' as timezone-aware datetime
# Pandas can parse 'YYYY-MM-DDTHH:MM+0100' directly.
df_pollutants["Datum"] = pd.to_datetime(df_pollutants["Datum"], errors="coerce")

# 2) Create date and time columns
df_pollutants["Datum_date"] = df_pollutants["Datum"].dt.date      # dtype: object (date objects)
df_pollutants["Datum_time"] = df_pollutants["Datum"].dt.time      # dtype: object (time objects)

# Optional: if you prefer pure pandas dtypes (no Python date/time objects),
# you can keep them as strings or as datetime64:
# df_pollutants["Datum_date"] = df_pollutants["Datum"].dt.strftime("%Y-%m-%d")
# df_pollutants["Datum_time"] = df_pollutants["Datum"].dt.strftime("%H:%M:%S%z")

# 3) Now pivot using the original 'Datum' or the split columns as needed
df_pollutants_pivot = df_pollutants.pivot(
    #index=["Datum", "Standort"], 
    index = ['Datum_date','Datum_time','Standort'], # you can also use ['Datum_date','Datum_time','Standort']
    columns="Parameter",
    values="Wert"
)

# 4) Drop unwanted columns (handle the HTML entity if present)
cols_to_drop = ["CO", "O3_max_h1", "O3_nb_h1>120", "SO2", "PN"] # in these two there are also many NAs:, "PM10", "PM2.5"]
df_pollutants_pivot = df_pollutants_pivot.drop(
    columns=[c for c in cols_to_drop if c in df_pollutants_pivot.columns],
    errors="ignore"
)

df_pollutants_pivot

Parameter                                         NO    NO2    NOx     O3  \
Datum_date Datum_time Standort                                              
2023-01-01 00:00:00   Zch_Heubeeribüel          0.71   4.27   2.80  51.47   
                      Zch_Rosengartenstrasse   34.90  36.30  46.96  18.24   
                      Zch_Schimmelstrasse      16.44  25.13  26.32  23.48   
                      Zch_Stampfenbachstrasse  16.57  26.47  27.13  20.00   
2023-01-02 00:00:00   Zch_Heubeeribüel          0.66   4.66   2.96  57.65   
...                                              ...    ...    ...    ...   
2024-12-30 00:00:00   Zch_Stampfenbachstrasse   9.24  34.43  25.41   6.90   
2024-12-31 00:00:00   Zch_Heubeeribüel          5.07  28.53  18.98  13.42   
                      Zch_Rosengartenstrasse   21.20  40.10  37.96   5.97   
                      Zch_Schimmelstrasse      18.14  38.21  34.53   7.71   
                      Zch_Stampfenbachstrasse   8.81  35.57  25.66   7.55   

Parameter                                       PM10  PM2.5  
Datum_date Datum_time Standort                               
2023-01-01 00:00:00   Zch_Heubeeribüel           NaN    NaN  
                      Zch_Rosengartenstrasse   16.22  14.55  
                      Zch_Schimmelstrasse      14.78  13.04  
                      Zch_Stampfenbachstrasse  22.99  17.52  
2023-01-02 00:00:00   Zch_Heubeeribüel           NaN    NaN  
...                                              ...    ...  
2024-12-30 00:00:00   Zch_Stampfenbachstrasse  16.72  14.94  
2024-12-31 00:00:00   Zch_Heubeeribüel         13.75  13.36  
                      Zch_Rosengartenstrasse   21.54  21.01  
                      Zch_Schimmelstrasse      18.54  18.57  
                      Zch_Stampfenbachstrasse  19.18  17.44  

[2924 rows x 6 columns]

In [ ]:
# Pivot of Selina
df_pollutants_pivot = df_pollutants.pivot(index=["Datum", "Standort"], columns="Parameter", values="Wert")
df_pollutants_pivot = df_pollutants_pivot.drop(columns=["CO", "O3_max_h1", "O3_nb_h1>120", "SO2", "PN"])

df_pollutants_pivot
#df_pollutants_pivot.head()
#df_pollutants_pivot.tail()

Parameter                                         NO    NO2    NOx     O3  \
Datum                 Standort                                              
2023-01-01T00:00+0100 Zch_Heubeeribüel          0.71   4.27   2.80  51.47   
                      Zch_Rosengartenstrasse   34.90  36.30  46.96  18.24   
                      Zch_Schimmelstrasse      16.44  25.13  26.32  23.48   
                      Zch_Stampfenbachstrasse  16.57  26.47  27.13  20.00   
2023-01-02T00:00+0100 Zch_Heubeeribüel          0.66   4.66   2.96  57.65   
...                                              ...    ...    ...    ...   
2024-12-30T00:00+0100 Zch_Stampfenbachstrasse   9.24  34.43  25.41   6.90   
2024-12-31T00:00+0100 Zch_Heubeeribüel          5.07  28.53  18.98  13.42   
                      Zch_Rosengartenstrasse   21.20  40.10  37.96   5.97   
                      Zch_Schimmelstrasse      18.14  38.21  34.53   7.71   
                      Zch_Stampfenbachstrasse   8.81  35.57  25.66   7.55   

Parameter                                       PM10  PM2.5  
Datum                 Standort                               
2023-01-01T00:00+0100 Zch_Heubeeribüel           NaN    NaN  
                      Zch_Rosengartenstrasse   16.22  14.55  
                      Zch_Schimmelstrasse      14.78  13.04  
                      Zch_Stampfenbachstrasse  22.99  17.52  
2023-01-02T00:00+0100 Zch_Heubeeribüel           NaN    NaN  
...                                              ...    ...  
2024-12-30T00:00+0100 Zch_Stampfenbachstrasse  16.72  14.94  
2024-12-31T00:00+0100 Zch_Heubeeribüel         13.75  13.36  
                      Zch_Rosengartenstrasse   21.54  21.01  
                      Zch_Schimmelstrasse      18.54  18.57  
                      Zch_Stampfenbachstrasse  19.18  17.44  

[2924 rows x 6 columns]

In [64]:
# Display df_pollutants info and missing values
print("df_pollutants shape:", df_pollutants_pivot.shape)
print("\nMissing values per column:")
print(df_pollutants_pivot.isnull().sum())
print("\nData types:")
print(df_pollutants_pivot.dtypes)

df_pollutants shape: (2924, 6)

Missing values per column:
Parameter
NO        78
NO2       78
NOx       78
O3        67
PM10     493
PM2.5    493
dtype: int64

Data types:
Parameter
NO       float64
NO2      float64
NOx      float64
O3       float64
PM10     float64
PM2.5    float64
dtype: object


In [ ]:
import pandas as pd

# 1) Boolean mask: any pollutant missing in a row (row = one index combination)
row_has_na = df_pollutants_pivot.isna().any(axis=1)

# 2) Build a long DataFrame of NA cells → includes the pollutant column
na_long = (
    df_pollutants_pivot
      .isna()
      .stack()                     # adds a "level_3" (pollutant) index level
      .rename("is_na")
      .reset_index()
)

# 3) Keep only the missing entries
na_long = na_long[na_long["is_na"]].drop(columns="is_na")

# 4) Rename the stacked column to "Parameter" (pollutant)
# The name of the stacked column depends on your frame; it's often "level_3" or similar.
stacked_col = [c for c in na_long.columns if c not in ["Datum_date", "Datum_time", "Standort"]][0]
na_long = na_long.rename(columns={stacked_col: "Parameter"})

# 5) Sort for readability
na_long = na_long.sort_values(["Datum_date", "Datum_time", "Standort", "Parameter"]).reset_index(drop=True)

# Now you have the four columns you want:
# Columns: Datum_date, Datum_time, Standort, Parameter
print(na_long.head(10))

# 6) Pretty print grouped by timestamp → list stations together with pollutants
from collections import defaultdict

by_dt = defaultdict(list)
for _, r in na_long.iterrows():
    key = (str(r["Datum_date"]), str(r["Datum_time"]))
    by_dt[key].append((str(r["Standort"]), str(r["Parameter"])))

print(f"{len(na_long)} missing cells (grouped):")
for (d, t) in sorted(by_dt.keys()):
    # aggregate per location: collect pollutants per Standort
    loc_to_poll = defaultdict(set)
    for loc, param in by_dt[(d, t)]:
        loc_to_poll[loc].add(param)
    parts = [f"{loc}: {', '.join(sorted(params))}" for loc, params in sorted(loc_to_poll.items())]
    print(f"{d} {t} → " + " | ".join(parts))

   Datum_date Datum_time          Standort Parameter
0  2023-01-01   00:00:00  Zch_Heubeeribüel      PM10
1  2023-01-01   00:00:00  Zch_Heubeeribüel     PM2.5
2  2023-01-02   00:00:00  Zch_Heubeeribüel      PM10
3  2023-01-02   00:00:00  Zch_Heubeeribüel     PM2.5
4  2023-01-03   00:00:00  Zch_Heubeeribüel      PM10
5  2023-01-03   00:00:00  Zch_Heubeeribüel     PM2.5
6  2023-01-04   00:00:00  Zch_Heubeeribüel      PM10
7  2023-01-04   00:00:00  Zch_Heubeeribüel     PM2.5
8  2023-01-05   00:00:00  Zch_Heubeeribüel      PM10
9  2023-01-05   00:00:00  Zch_Heubeeribüel     PM2.5
1287 missing cells (grouped):
2023-01-01 00:00:00 → Zch_Heubeeribüel: PM10, PM2.5
2023-01-02 00:00:00 → Zch_Heubeeribüel: PM10, PM2.5
2023-01-03 00:00:00 → Zch_Heubeeribüel: PM10, PM2.5
2023-01-04 00:00:00 → Zch_Heubeeribüel: PM10, PM2.5
2023-01-05 00:00:00 → Zch_Heubeeribüel: PM10, PM2.5
2023-01-06 00:00:00 → Zch_Heubeeribüel: PM10, PM2.5
2023-01-07 00:00:00 → Zch_Heubeeribüel: PM10, PM2.5
2023-01-08 00:00:00 → Z

## Traffic Data

In [39]:
df_traffic = pd.read_csv("data/traffic_data_2023_2024.csv")
df_traffic.head()

,ZSName,MessungDatZeit,AnzFahrzeuge,year,pollutant_station
0,Rosengartenstrasse (Nordstrasse),2023-01-01T00:00:00,688.0,2023,Zch_Rosengartenstrasse
1,Rosengartenstrasse (Nordstrasse),2023-01-01T01:00:00,1370.0,2023,Zch_Rosengartenstrasse
2,Rosengartenstrasse (Nordstrasse),2023-01-01T02:00:00,1041.0,2023,Zch_Rosengartenstrasse
3,Rosengartenstrasse (Nordstrasse),2023-01-01T03:00:00,661.0,2023,Zch_Rosengartenstrasse
4,Rosengartenstrasse (Nordstrasse),2023-01-01T04:00:00,528.0,2023,Zch_Rosengartenstrasse


In [86]:
# 1) Parse MessungDatZeit safely into datetime
df_traffic["MessungDatZeit"] = pd.to_datetime(
    df_traffic["MessungDatZeit"], errors="coerce"  # handles bad rows as NaT
)

# 2) If your timestamps have timezone offsets and you want to remove tz info, uncomment:
# df_traffic["MessungDatZeit"] = df_traffic["MessungDatZeit"].dt.tz_localize(None)

# 3) Create separate columns for date & time
df_traffic["Messung_date"] = df_traffic["MessungDatZeit"].dt.date     # dtype: object (datetime.date)
df_traffic["Messung_time"] = df_traffic["MessungDatZeit"].dt.time     # dtype: object (datetime.time)

# rename pollutant_station to Standort to match df_pollutants
df_traffic = df_traffic.rename(columns={"pollutant_station": "Standort"})

# delete ZSName column
df_traffic = df_traffic.drop(columns=["ZSName"], errors="ignore")

# reorder columns
column_order = ["year", "Messung_date", "Messung_time", "MessungDatZeit", "Standort", "AnzFahrzeuge"]
df_traffic = df_traffic[column_order]

# (Optional) If you prefer **string** columns (sometimes easier to export/sort):
# df_traffic["messung_date"] = df_traffic["MessungDatZeit"].dt.strftime("%Y-%m-%d")
# df_traffic["messung_time"] = df_traffic["MessungDatZeit"].dt.strftime("%H:%M:%S")

df_traffic

,year,Messung_date,Messung_time,MessungDatZeit,Standort,AnzFahrzeuge
0,2023,2023-01-01,00:00:00,2023-01-01 00:00:00,Zch_Rosengartenstrasse,688.0
1,2023,2023-01-01,01:00:00,2023-01-01 01:00:00,Zch_Rosengartenstrasse,1370.0
2,2023,2023-01-01,02:00:00,2023-01-01 02:00:00,Zch_Rosengartenstrasse,1041.0
3,2023,2023-01-01,03:00:00,2023-01-01 03:00:00,Zch_Rosengartenstrasse,661.0
4,2023,2023-01-01,04:00:00,2023-01-01 04:00:00,Zch_Rosengartenstrasse,528.0
...,...,...,...,...,...,...
119947,2024,2024-12-31,19:00:00,2024-12-31 19:00:00,Zch_Heubeeribüel,14.0
119948,2024,2024-12-31,20:00:00,2024-12-31 20:00:00,Zch_Heubeeribüel,14.0
119949,2024,2024-12-31,21:00:00,2024-12-31 21:00:00,Zch_Heubeeribüel,15.0
119950,2024,2024-12-31,22:00:00,2024-12-31 22:00:00,Zch_Heubeeribüel,33.0


In [82]:
# Display df_traffic info and missing values
print("df_traffic shape:", df_traffic.shape)
print("\nMissing values per column:")
print(df_traffic.isnull().sum())
print("\nData types:")
print(df_traffic.dtypes)

df_traffic shape: (119952, 6)

Missing values per column:
year                 0
messung_date         0
messung_time         0
MessungDatZeit       0
Standort             0
AnzFahrzeuge      1010
dtype: int64

Data types:
year                       int64
messung_date              object
messung_time              object
MessungDatZeit    datetime64[ns]
Standort                  object
AnzFahrzeuge             float64
dtype: object


In [ ]:
from collections import defaultdict

# Build your triplets
triplets = (
    df_traffic.loc[missing_mask, ["MessungDatZeit", "Standort"]]
      .dropna(subset=["MessungDatZeit"])
      .assign(
          date_str=lambda x: x["MessungDatZeit"].dt.strftime("%Y-%m-%d"),
          time_str=lambda x: x["MessungDatZeit"].dt.strftime("%H:%M:%S")
      )
      [["date_str", "time_str", "Standort"]]
      .values
      .tolist()
)

# 2) Sort by date, then time, then location
triplets.sort(key=lambda t: (t[0], t[1], t[2]))

# 3) Group locations per (date, time)
by_dt = defaultdict(list)
for date_str, time_str, loc in triplets:
    by_dt[(date_str, time_str)].append(loc)

# 4) Print in order (date, time), with locations joined
print(f"{len(triplets)} rows with missing 'AnzFahrzeuge' (grouped):")
for (date_str, time_str) in sorted(by_dt.keys()):
    locs = ", ".join(sorted(set(by_dt[(date_str, time_str)])))  # unique + sorted
    print(f"{date_str} {time_str} → {locs}")

1010 rows with missing 'AnzFahrzeuge' (grouped):
2023-02-13 22:00:00 → Zch_Stampfenbachstrasse
2023-02-13 23:00:00 → Zch_Stampfenbachstrasse
2023-02-16 21:00:00 → Zch_Stampfenbachstrasse
2023-02-16 22:00:00 → Zch_Stampfenbachstrasse
2023-03-13 21:00:00 → Zch_Stampfenbachstrasse
2023-03-13 22:00:00 → Zch_Stampfenbachstrasse
2023-03-13 23:00:00 → Zch_Stampfenbachstrasse
2023-03-14 00:00:00 → Zch_Stampfenbachstrasse
2023-03-22 10:00:00 → Zch_Rosengartenstrasse
2023-03-22 11:00:00 → Zch_Rosengartenstrasse
2023-05-05 17:00:00 → Zch_Rosengartenstrasse
2023-05-05 18:00:00 → Zch_Rosengartenstrasse
2023-05-05 19:00:00 → Zch_Rosengartenstrasse
2023-05-05 20:00:00 → Zch_Rosengartenstrasse
2023-12-13 13:00:00 → Zch_Schimmelstrasse
2023-12-13 14:00:00 → Zch_Schimmelstrasse
2024-01-08 00:00:00 → Zch_Rosengartenstrasse, Zch_Schimmelstrasse, Zch_Stampfenbachstrasse
2024-01-08 01:00:00 → Zch_Rosengartenstrasse, Zch_Schimmelstrasse, Zch_Stampfenbachstrasse
2024-01-08 02:00:00 → Zch_Rosengartenstrasse, Z

In [ ]:
# Which station-day groups have no valid hourly values for the whole day? -> not possible to calculate daily mean (see error next cell)
g = df_traffic.groupby(["Standort", "Messung_date"])["AnzFahrzeuge"]

summary = (
    g.agg(
        valid_hours=lambda s: s.notna().sum(),
        total_hours="size",
        daily_mean="mean"
    )
    .reset_index()
    .sort_values(["Messung_date", "Standort"])
)

# These are the groups that will produce NaN means (and cause IntCastingNaNError if you cast to int)
all_missing_groups = summary[summary["valid_hours"] == 0]

print(f"# station-day groups with ALL hourly values missing: {len(all_missing_groups)}")
display(all_missing_groups.head(20))


# station-day groups with ALL hourly values missing: 17


,Standort,Messung_date,valid_hours,total_hours,daily_mean
2167,Zch_Stampfenbachstrasse,2024-02-04,0,48,NaN
140,Zch_Heubeeribüel,2024-07-19,0,48,NaN
871,Zch_Rosengartenstrasse,2024-07-19,0,48,NaN
1602,Zch_Schimmelstrasse,2024-07-19,0,48,NaN
2333,Zch_Stampfenbachstrasse,2024-07-19,0,48,NaN
141,Zch_Heubeeribüel,2024-07-20,0,48,NaN
872,Zch_Rosengartenstrasse,2024-07-20,0,48,NaN
1603,Zch_Schimmelstrasse,2024-07-20,0,48,NaN
2334,Zch_Stampfenbachstrasse,2024-07-20,0,48,NaN
142,Zch_Heubeeribüel,2024-07-21,0,48,NaN


=> delete dates 19.07.2024 - 22.07.2024 for  4 out of 5 stations + additionally 04.02.2024 for one of the five stations

In [104]:

daily_avg = (
    df_traffic
    .groupby(["Standort", "Messung_date"], as_index=False)["AnzFahrzeuge"]
    .mean()
    .round(0)  # round to nearest integer
    .astype({"AnzFahrzeuge": "int"})  # convert to integer type
    .rename(columns={"AnzFahrzeuge": "AnzFahrzeuge_daily_mean"})
)

daily_avg.head()


IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer: Error while type casting for column 'AnzFahrzeuge'

In [ ]:
# FIX: delete station-days with all missing hourly values before averaging & rounding
# 1) Daily means + coverage from the original hourly data
daily_qc = (
    df_traffic
    .groupby(["Standort", "Messung_date"])["AnzFahrzeuge"]
    .agg(
        daily_mean="mean",
        valid_hours=lambda s: s.notna().sum(),
        total_hours="size"
    )
    .reset_index()
)

# 2) Exclude full-day outages
daily_kept = daily_qc[daily_qc["valid_hours"] > 0].copy()

# Optional: enforce a minimum coverage (e.g., require ≥ 18 hours)
# min_hours = 18
# daily_kept = daily_kept[daily_kept["valid_hours"] >= min_hours]

# 3) Round the daily mean safely
daily_kept["AnzFahrzeuge_daily_mean"] = (
    pd.Series(daily_kept["daily_mean"]).round(0).astype("Int64")
)

# 4) Average per station over the retained days
station_avg = (
    daily_kept
    .groupby("Standort", as_index=False)["AnzFahrzeuge_daily_mean"]
    .mean()  # could also do median() ...
    .round(0)
    .astype({"AnzFahrzeuge_daily_mean": "Int64"})
    .rename(columns={"AnzFahrzeuge_daily_mean": "AnzFahrzeuge_station_mean"})
)

display(daily_kept)
display(station_avg)

,Standort,Messung_date,daily_mean,valid_hours,total_hours,AnzFahrzeuge_daily_mean
0,Zch_Heubeeribüel,2024-03-01,53.916667,48,48,54
1,Zch_Heubeeribüel,2024-03-02,81.937500,48,48,82
2,Zch_Heubeeribüel,2024-03-03,84.625000,48,48,85
3,Zch_Heubeeribüel,2024-03-04,64.000000,48,48,64
4,Zch_Heubeeribüel,2024-03-05,51.000000,48,48,51
...,...,...,...,...,...,...
2494,Zch_Stampfenbachstrasse,2024-12-27,206.395833,48,48,206
2495,Zch_Stampfenbachstrasse,2024-12-28,180.291667,48,48,180
2496,Zch_Stampfenbachstrasse,2024-12-29,149.020833,48,48,149
2497,Zch_Stampfenbachstrasse,2024-12-30,187.937500,48,48,188


,Standort,AnzFahrzeuge_station_mean
0,Zch_Heubeeribüel,67
1,Zch_Rosengartenstrasse,975
2,Zch_Schimmelstrasse,612
3,Zch_Stampfenbachstrasse,257
